[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/39_ppo_loss.ipynb)

# 🔴 Hard: PPO Clipped Loss

Implement the **PPO (Proximal Policy Optimization)** **clipped surrogate loss**.

Given:
- `new_logps`: current policy log-probs $(B,)$
- `old_logps`: old policy log-probs $(B,)$
- `advantages`: advantage estimates $(B,)$

Define the ratio

$$ r_i = \exp(\text{new\_logps}_i - \text{old\_logps}_i). $$

Then compute
- $L^{\text{unclipped}}_i = r_i A_i$
- $L^{\text{clipped}}_i = \operatorname{clip}(r_i, 1-\epsilon, 1+\epsilon) A_i$

The loss is the negative batch mean of the elementwise minimum:

$$
\mathcal{L}_\text{PPO} = -\mathbb{E}_i\big[\min(L^{\text{unclipped}}_i, L^{\text{clipped}}_i)\big].
$$

Implementation notes: detach `old_logps` and `advantages` so gradients only flow through `new_logps`.

### Signature
```python
from torch import Tensor

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    """PPO clipped surrogate loss over a batch."""
```


In [71]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [72]:
import torch
import torch.nn.functional as F
from torch import Tensor


In [25]:
# ✏️ YOUR IMPLEMENTATION HERE

def ppo_loss(new_logps: Tensor, old_logps: Tensor, advantages: Tensor,
             clip_ratio: float = 0.2) -> Tensor:
    diff_logps = torch.exp(new_logps - old_logps.detach())
    loss = torch.minimum(diff_logps * advantages.detach(), torch.clamp(diff_logps, min=1 - clip_ratio, max=1 + clip_ratio) * advantages.detach())
    return loss.mean()*-1.0
    pass  # -mean(min(r * adv, clamp(r, 1-clip, 1+clip) * adv)) with gradients only through new_logps


In [26]:
# 🧪 Debug
new_logps = torch.tensor([0.0, -0.2, -0.4, -0.6])
old_logps = torch.tensor([0.0, -0.1, -0.5, -0.5])
advantages = torch.tensor([1.0, -1.0, 0.5, -0.5])
print('Loss:', ppo_loss(new_logps, old_logps, advantages, clip_ratio=0.2))


Loss: tensor(-0.0488)


In [27]:
# ✅ SUBMIT
from torch_judge import check
check('ppo_loss')



🧪 Testing: PPO (Proximal Policy Optimization) Clipped Loss (Hard)
──────────────────────────────────────────────────
  ✅ [1/3] Basic shape & type (1.1ms)
  ✅ [2/3] Numeric check vs fixed value (1.3ms)
  ✅ [3/3] Gradient flows to new_logps only (0.8ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (3.2ms total)
  Progress saved. Run status() to see your dashboard.



In [83]:
def compute_gae(rewards, action_mask, values, gamma, lam):
    # rewards: (B, 1) scalar terminal reward   action_mask,values: (B, S)
    # last_idx = action_mask.long().cumsum(-1).argmax(-1, keepdim=True)
    # done = db,t​=1[t≥jb​]    # 1 at terminal AND after
    # scatter `rewards` into zeros(B,S) at last_idx
    # reverse loop: running (B,), next_values (B,), init 0
    # return advantages * action_mask                 # (B, S), advantages only
    ...
    last_index = torch.argmax(torch.cumsum(action_mask.long(), dim=-1), dim=-1, keepdim=True)
    reward_scattered = torch.zeros(action_mask.shape)
    reward_scattered.scatter_(-1, last_index, rewards)
    done_t = (torch.arange(action_mask.shape[1]).unsqueeze(0) >= last_index).int()
    V_next = torch.zeros(rewards.shape).squeeze()
    g = torch.zeros(rewards.shape).squeeze()
    advantage = torch.zeros(action_mask.shape)
    for t in range(action_mask.shape[1]-1, -1, -1):
      delta = reward_scattered[:,t] + gamma * (1  - done_t[:,t]) * V_next - values[:,t]
      g = delta + gamma * lam * (1  - done_t[:,t]) * g
      advantage[:,t] = g
      V_next = values[:,t]
    return advantage * action_mask


In [85]:
import torch

def _test_compute_gae():
    # ---- Test 1: shapes and dtype ----
    B, S = 2, 4
    mask = torch.tensor([[1.,1.,1.,1.],[1.,1.,0.,0.]])
    values = torch.zeros(B, S)
    rewards = torch.tensor([[1.0],[1.0]])
    A = compute_gae(rewards, mask, values, gamma=0.99, lam=0.95)
    assert A.shape == (B, S), A.shape
    assert A.dtype == torch.float32
    print("✓ 1: shape and dtype")

    # ---- Test 2: padding advantages are exactly zero ----
    assert torch.allclose(A[1, 2:], torch.zeros(2)), A[1]
    print("✓ 2: padding zeroed")

    # ---- Test 3: terminal token has no bootstrap, for ANY gamma/lam ----
    # Sentinels go in row1's REAL padding (idx 2,3); row0 is fully active so its
    # terminal value must be a real number.
    values = torch.tensor([[0.5, 1.0, 2.0, 2.0],   # row0 fully active, terminal idx3 V=2.0
                           [0.5, 1.0, 9.9, 9.9]])   # row1 padding at idx2,3 (must not leak)
    rewards = torch.tensor([[3.0],[3.0]])
    for g, l in [(1.0, 1.0), (0.99, 0.95), (0.5, 0.3)]:
        A = compute_gae(rewards, mask, values, gamma=g, lam=l)
        assert torch.allclose(A[0, 3], torch.tensor(3.0 - 2.0), atol=1e-5), (g, l, A[0, 3])
        assert torch.allclose(A[1, 1], torch.tensor(3.0 - 1.0), atol=1e-5), (g, l, A[1, 1])
    print("✓ 3: terminal drops the bootstrap, any gamma/lam; padding does not leak")

    # ---- Test 4: hand-computed full trajectory (row0, all 4 active) ----
    mask4 = torch.tensor([[1.,1.,1.,1.]])
    values4 = torch.tensor([[0.5, 1.0, 2.0, 4.0]])
    R = 3.0
    rewards4 = torch.tensor([[R]])
    A = compute_gae(rewards4, mask4, values4, gamma=1.0, lam=1.0)
    # reward after scatter = [0,0,0,3], done only at t=3, gamma=lam=1:
    # t=3: delta = 3 + 0     - 4.0 = -1.0   A3 = -1.0
    # t=2: delta = 0 + 4.0   - 2.0 =  2.0   A2 = 2.0 + A3 = 1.0
    # t=1: delta = 0 + 2.0   - 1.0 =  1.0   A1 = 1.0 + A2 = 2.0
    # t=0: delta = 0 + 1.0   - 0.5 =  0.5   A0 = 0.5 + A1 = 2.5
    expected = torch.tensor([[2.5, 2.0, 1.0, -1.0]])
    assert torch.allclose(A, expected, atol=1e-5), A
    print("✓ 4: matches hand-computed GAE")

    # ---- Test 5: gamma=lam=1 => returns collapse to the Monte-Carlo return R ----
    ret = A + values4
    assert torch.allclose(ret[mask4.bool()], torch.full((int(mask4.sum()),), R), atol=1e-5), ret
    print("✓ 5: gamma=lam=1 gives returns == total reward everywhere")

    # ---- Test 6: gamma=lam=0 => A = (scattered reward - value) on active tokens ----
    A = compute_gae(rewards4, mask4, values4, gamma=0.0, lam=0.0)
    expected = torch.tensor([[0-0.5, 0-1.0, 0-2.0, R-4.0]])
    assert torch.allclose(A, expected, atol=1e-5), A
    print("✓ 6: gamma=lam=0 reduces to reward-minus-baseline")

    print("\nAll compute_gae tests passed.")

_test_compute_gae()

✓ 1: shape and dtype
✓ 2: padding zeroed
✓ 3: terminal drops the bootstrap, any gamma/lam; padding does not leak
✓ 4: matches hand-computed GAE
✓ 5: gamma=lam=1 gives returns == total reward everywhere
✓ 6: gamma=lam=0 reduces to reward-minus-baseline

All compute_gae tests passed.
